# Strands Agents with Bedrock AgentCore Browser

This lab demonstrates how to use Amazon Bedrock AgentCore Browser to navigate financial websites and extract regulatory data.

## Overview

In this lab, you will:
- Create a custom browser with public network access
- Navigate the Reserve Bank of Australia (RBA) website
- Extract monetary policy information
- Read RBA Board meeting minutes

## Why Browser Automation for FSI?

- **Regulatory monitoring** — Check central bank decisions, policy changes
- **Data extraction** — Pull rates, statistics from financial portals
- **Compliance evidence** — Automated proof of monitoring activities
- **Legacy systems** — Interact with web apps that have no API

## Prerequisites

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore playwright

In [5]:
import boto3
region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## What is Bedrock AgentCore Browser?

Amazon Bedrock AgentCore Browser is a powerful tool that enables AI agents to interact with web browsers dynamically in a secure, managed environment. Key capabilities include:

- **Web Navigation**: Navigate to websites, click elements, and fill forms programmatically
- **Content Extraction**: Extract information from web pages and capture screenshots
- **Secure Environment**: Runs in an isolated, secure browser environment
- **JavaScript Execution**: Execute custom JavaScript for advanced web interactions
- **Session Management**: Maintain browser sessions across multiple operations

The Browser tool enables agents to perform complex web-based tasks that require visual understanding and interactive capabilities.


## Step 1: Create a Custom Browser with Public Network

The default browser has restrictive settings. For accessing external financial sites, we create a **custom browser** with public network access.

In [6]:
from bedrock_agentcore._utils import endpoints
import boto3
from botocore.exceptions import ClientError

region = boto3.session.Session().region_name
cp_endpoint = endpoints.get_control_plane_endpoint(region)

cp_client = boto3.client('bedrock-agentcore-control', region_name=region, endpoint_url=cp_endpoint)

browser_name = 'fsi_regulatory_browser'
try:
    response = cp_client.create_browser(
        name=browser_name,
        description='Custom browser for FSI regulatory site monitoring',
        networkConfiguration={'networkMode': 'PUBLIC'},
    )
    browser_id = response['browserId']
    print(f'✅ Custom browser created: {browser_id}')
except ClientError as e:
    if 'already exists' in str(e).lower() or 'Conflict' in str(e):
        browsers = cp_client.list_browsers()['browserSummaries']
        browser_id = next(b['browserId'] for b in browsers if b.get('name') == browser_name)
        print(f'✅ Using existing browser: {browser_id}')
    else:
        raise e

print(f'   Network: PUBLIC (can access any website)')

✅ Using existing browser: fsi_regulatory_browser-TOPzdqpjcy
   Network: PUBLIC (can access any website)


## Step 2: Navigate RBA — Cash Rate Target

Let's navigate to the Reserve Bank of Australia and extract information about the current cash rate.

In [3]:
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright
from strands import Agent
from strands.models import BedrockModel

with browser_session(region, identifier=browser_id) as client:
    print(f"🌐 Session: {client.session_id}")
    ws_url, headers = client.generate_ws_headers()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        print("✅ Browser connected")

        context = browser.contexts[0] if browser.contexts else await browser.new_context()
        page = context.pages[0] if context.pages else await context.new_page()

        # Navigate to RBA Cash Rate page
        await page.goto("https://www.rba.gov.au/statistics/cash-rate/", timeout=30000)
        await page.wait_for_load_state("networkidle")

        title = await page.title()
        content = await page.inner_text("body")
        print(f"📄 Page loaded: {title}")

        await browser.close()

# Now use the agent to summarize the extracted content
summarizer = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt="You are a financial analyst. Summarize regulatory content concisely.",
)

summarizer(f"Summarize the key information from this RBA Cash Rate page in 3-4 bullet points:\n\n{content[:2000]}")


🌐 Session: 01KT0KFWC2BAZ6QEC3HY6TGRJP
✅ Browser connected

📄 Page: Cash Rate Target | RBA

--- Content Preview ---
  About Us
  Media Releases
  Speeches
  Publications
  Statistics
  Chart Pack
  Research
  Archives
  Education
  Careers
  Q&A
  Contact Us
  You are here:
  Home Statistics Cash Rate Target
  In Statistics


## Step 3: Read RBA Board Meeting Minutes

Let's navigate to the latest Monetary Policy Board meeting minutes and extract the key decisions.

In [4]:
with browser_session(region, identifier=browser_id) as client:
    print(f"🌐 Session: {client.session_id}")
    ws_url, headers = client.generate_ws_headers()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        context = browser.contexts[0] if browser.contexts else await browser.new_context()
        page = context.pages[0] if context.pages else await context.new_page()

        # Navigate to the 2026 board minutes index page
        await page.goto("https://www.rba.gov.au/monetary-policy/rba-board-minutes/2026/", timeout=30000)
        await page.wait_for_load_state("networkidle")

        # Find the most recent meeting minutes link
        links = await page.query_selector_all("a[href*='2026/2026-']")
        if links:
            last_link = links[-1]
            href = await last_link.get_attribute("href")
            print(f"📄 Most recent minutes: {href}")
            await last_link.click()
            await page.wait_for_load_state("networkidle")
        else:
            # Fallback: navigate directly to latest known
            await page.goto("https://www.rba.gov.au/monetary-policy/rba-board-minutes/2026/2026-05-05.html", timeout=30000)
            await page.wait_for_load_state("networkidle")

        title = await page.title()
        content = await page.inner_text("body")
        print(f"📄 Page: {title}")

        # Show a preview of the meeting content
        lines = [l.strip() for l in content.split("\n") if l.strip()]
        start = next((i for i, l in enumerate(lines) if "Members participating" in l), 20)
        print("\n--- Meeting Minutes Preview ---")
        for line in lines[start:start+15]:
            print(f"  {line}")

        await browser.close()

# Summarize with the agent
summarizer(f"Summarize the key decisions from this RBA Board meeting in 3-5 bullet points:\n\n{content[:3000]}")


🌐 Session: 01KT0KGF9HGV5VKC6BQBCXVE04
📄 Page: 20 May 2025 | Minutes of the Monetary Policy Board Meeting | RBA

--- Meeting Minutes Preview ---
  Members participating
  Michele Bullock (Governor and Chair), Andrew Hauser (Deputy Governor and Deputy Chair), Marnie Baker, Renée Fry‑McKibbin, Ian Harper AO, Carolyn Hewson AO, Steven Kennedy PSM, Iain Ross AO, Alison Watkins AM
  Others participating
  Sarah Hunter (Assistant Governor, Economic), Christopher Kent (Assistant Governor, Financial Markets)
  Anthony Dickman (Secretary), David Norman (Deputy Secretary)
  Meredith Beechey Osterholm (Head, Monetary Policy Strategy), Sally Cray (Chief Communications Officer), David Jacobs (Head, Domestic Markets Department), Michael Plumb (Head, Economic Analysis Department), Penelope Smith (Head, International Department)
  Financial conditions
  Members began their discussion by considering the evolving news on US tariff policy and its impact on global financial markets. The tariffs announced b

## Step 4: Use AI to Summarize the Page Content

Now let's combine the browser (for fetching) with a Strands Agent (for summarizing). The browser extracts the raw text, and the agent summarizes it.

In [5]:
from strands import Agent, tool
from strands.models import BedrockModel

# First, fetch the page content with the browser
with browser_session(region, identifier=browser_id) as client:
    ws_url, headers = client.generate_ws_headers()
    async with async_playwright() as playwright:
        browser = await playwright.chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        context = browser.contexts[0] if browser.contexts else await browser.new_context()
        page = context.pages[0] if context.pages else await context.new_page()

        await page.goto('https://www.rba.gov.au/monetary-policy/rba-board-minutes/2025/2025-05-20.html', timeout=30000)
        await page.wait_for_load_state('networkidle')
        page_content = await page.inner_text('body')
        await browser.close()

print(f'Fetched {len(page_content)} characters from RBA Board Minutes')

# Now use a Strands Agent to summarize
summarizer = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are a financial analyst. Summarize regulatory documents concisely.',
)

summarizer(f'Summarize the key decisions from this RBA Monetary Policy Board meeting in 3-5 bullet points:\n\n{page_content[:3000]}')

Fetched 29055 characters from RBA Board Minutes
Here are the key decisions from the RBA Monetary Policy Board meeting held on 19 and 20 May 2025:

- The Board discussed the impact of recent US tariff policy on global financial markets, noting the initial turbulence and subsequent stabilization.
- Market expectations for central bank policy rates in advanced economies had declined initially but had not fully recovered, with most markets expecting continued rate cuts.
- Longer-term government bond yields in advanced economies, particularly in the United States, were generally higher than at the previous meeting.
- The Board considered the implications of these financial conditions for Australia's economic outlook and monetary policy.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here are the key decisions from the RBA Monetary Policy Board meeting held on 19 and 20 May 2025:\n\n- The Board discussed the impact of recent US tariff policy on global financial markets, noting the initial turbulence and subsequent stabilization.\n- Market expectations for central bank policy rates in advanced economies had declined initially but had not fully recovered, with most markets expecting continued rate cuts.\n- Longer-term government bond yields in advanced economies, particularly in the United States, were generally higher than at the previous meeting.\n- The Board considered the implications of these financial conditions for Australia's economic outlook and monetary policy."}], 'metadata': {'usage': {'inputTokens': 759, 'outputTokens': 127, 'totalTokens': 886}, 'metrics': {'latencyMs': 885, 'timeToFirstByteMs': 447}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_du

## Cleanup (Optional)

In [ ]:
# Uncomment to delete the custom browser
# cp_client.delete_browser(browserId=browser_id)
# print('✅ Browser deleted')

## Summary

In this lab, you:

- ✅ Created a custom browser with public network access
- ✅ Navigated the RBA website and extracted cash rate information
- ✅ Read Monetary Policy Board meeting minutes
- ✅ Combined browser (fetch) + AI agent (summarize) for regulatory analysis

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Custom browser with public network | Access any regulatory site |
| Playwright automation | Reliable, scriptable page interaction |
| Browser + Agent combo | Fetch unstructured data, then AI summarizes |
| Session isolation | Each session runs in its own secure microVM |

### Architecture Pattern

```
AgentCore Browser (fetch page) → Raw text → Strands Agent (summarize/analyze)
```

This pattern is more reliable than having the agent drive the browser directly, and gives you full control over what pages are accessed.

### Next: Lab 04 — AgentCore Runtime MCP
We'll deploy a transaction validation tool as a managed MCP server with authentication.